<a href="https://colab.research.google.com/github/hyperdbio/Android-inject/blob/master/ex09_RNN_%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0%ED%95%99%EC%8A%B5%ED%95%98%EA%B8%B0(%EA%B0%90%EC%84%B1%EB%B6%84%EC%84%9D).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 목표
- 네이버영화리뷰데이터를 기반으로 감성 분석 실습을 진행해보자
- RNN 계열 모델을 이해해보자
- Text 데이터 처리 방법에 대해 이해해보자

In [4]:
# gpu 연결
# 드라이브 연동
%cd '/content/drive/MyDrive/머신러닝'

/content/drive/MyDrive/머신러닝


### 데이터 불러오기
 - import
 - txt -> read_csv(), '\t' 구분자

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [9]:
# delimiter='\t'
train = pd.read_csv('./data/ratings_train.txt', delimiter='\t')
train

test = pd.read_csv('./data/ratings_test.txt', delimiter='\t')
test


,id,document,label
0,6270596,굳 ㅋ,1
1,9274899,GDNTOPCLASSINTHECLUB,0
2,8544678,뭐야 이 평점들은.... 나쁘진 않지만 10점 짜리는 더더욱 아니잖아,0
3,6825595,지루하지는 않은데 완전 막장임... 돈주고 보기에는....,0
4,6723715,3D만 아니었어도 별 다섯 개 줬을텐데.. 왜 3D로 나와서 제 심기를 불편하게 하죠??,0
...,...,...,...
49995,4608761,오랜만에 평점 로긴했네ㅋㅋ 킹왕짱 쌈뽕한 영화를 만났습니다 강렬하게 육쾌함,1
49996,5308387,의지 박약들이나 하는거다 탈영은 일단 주인공 김대희 닮았고 이등병 찐따 OOOO,0
49997,9072549,그림도 좋고 완성도도 높았지만... 보는 내내 불안하게 만든다,0
49998,5802125,절대 봐서는 안 될 영화.. 재미도 없고 기분만 잡치고.. 한 세트장에서 다 해먹네,0


In [11]:
# 데이터 크기 확인
train.shape, test.shape
# id: 사용자(고객) 번호(정보)
# document: 리뷰(텍스트) -> 문제 데이터
# label: 정답데이터 (0 부정, 1 긍정)

((150000, 3), (50000, 3))

In [12]:
# 데이터 정보확인
train.info()

# 리뷰 텍스트에 결측치 5개 존재함 -> 삭제 or 채움
# 특정행에 결측치가 있다면 해당 행은 삭제하겠다! (선택)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        150000 non-null  int64 
 1   document  149995 non-null  object
 2   label     150000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.4+ MB


### 데이터 전처리

In [ ]:
# 1. 결측치 처리
# inplace=True 결측치 있는 행을 삭제하고, 원본 변수에다가 처리된 데이터 다시 대입
train.dropna(inplace=True)
# train = train.dropna()

In [13]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        150000 non-null  int64 
 1   document  149995 non-null  object
 2   label     150000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.4+ MB


In [16]:
# test도 확인 후, 결측치가 존재하면 해당 행을 삭제해보자!
test.info() # 결측치 3 -> 제거
test.dropna(inplace=True)
# test2 = test.dropna()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        50000 non-null  int64 
 1   document  49997 non-null  object
 2   label     50000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 1.1+ MB


In [17]:
# 2. 데이터 분리
# train, test
# 문제(document), 답(label)
X_train = train['document']
y_train = train['label']
X_test = test['document']
y_test = test['label']

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(150000,) (150000,)
(49997,) (49997,)


In [19]:
X_train.head()

# 텍스트를 pc가 이해할 수 있는 형태로 변형
# 토큰화 -> 수치화

,document
0,아 더빙.. 진짜 짜증나네요 목소리
1,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나
2,너무재밓었다그래서보는것을추천한다
3,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정
4,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...


#### 텍스트 데이터 전처리
- 토큰화, 수치화
- 토큰화: 문서, 문장 특정 기준에 따라 맞춰서 잘라는 행위
      - 글자단위
      - 단어단위(띄어쓰기단위)
      - 형태소단위(명사, 형용사, 동사, 조사, ...): konlpy, kiwi 도구
- 수치화: 잘라낸 토큰을 일정기준에 맞춰서 숫자형태로 변경하는 행위
    - 원핫인코딩, BOW(Bag of Words), TF-IDF
    - 단어빈도 -> Countvectorizer() 클래스
    - 워드임베딩 -> 학습기반으로 수치화하는 방법(데이터의 의미를 담아줄 수 있음, 문맥의 의미를 연결시킬 수 있음)

In [21]:
# 문장 학습용 데이터를 수치로 변경해주는 클래스(도구)
# 쪼개줌(토큰화), 수치화 => 벡터화
from tensorflow.keras.layers import TextVectorization

In [23]:
# 벡터화도구 생성
vect = TextVectorization(max_tokens=5000,
                         output_mode='int',
                         standardize='lower_and_strip_punctuation',
                         output_sequence_length=10)
vect
# X_train, X_test 해당 도구를 통해 데이터 전처리 예정

<TextVectorization name=text_vectorization_1, built=False>